# Point3d API Documentation and Examples

This notebook demonstrates the **`Point3d`** and **`p3d_leanest`** classes from the `upxo.geoEntities` module. It provides comprehensive examples of:

- **Lightweight vs. full-featured point representations** – Compare memory-efficient `p3d_leanest` with the full-featured `Point3d` class for performance-critical applications
- **Equality comparisons** – Benchmark `==`, `.eq()`, and `.eq_fast()` methods across different point formats including lean points, full Point3d instances, tuples, lists, and numpy arrays, with configurable `point_spec` parameters
- **Geometric operations** – Demonstrate plane intersections via `from_three_planes()`, squared and Euclidean distance calculations with vectorized numpy support, and translation operations along direction vectors
- **Performance optimization** – Utilize `point_spec` parameters (1-8) to bypass validation layers and leverage optimized code paths for various input types
- **Neighbor search algorithms** – Implement k-nearest neighbor queries with `find_neigh_point_by_count()`, showcasing cross-dimensional handling between 2D and 3D coordinate systems

Each section includes detailed parameter documentation, development phase notes, performance benchmarks with large datasets (100,000+ points), and edge cases to guide selection of appropriate APIs for computational geometry workflows.

## Point3d vs `p3d_leanest` quickstart
Load both the full-featured `Point3d` and minimal `p3d_leanest`, plus numpy/matplotlib, so each subsequent cell can contrast API surface and memory-light usage.

In [25]:
from upxo.geoEntities.point3d import Point3d as p3d
from upxo.geoEntities.point3d import p3d_leanest
import numpy as np
import matplotlib.pyplot as plt

### Create lean point3d

Create two lean 3D points to illustrate the lightweight constructor and list wrapping—useful when you need many points without the heavier methods of `Point3d`.

In [26]:
a = [p3d_leanest(1, 2, 0), p3d_leanest(1, 2, 0)]
print(a)

[Lean 3D point at (1, 2, 0, 3180679338944), Lean 3D point at (1, 2, 0, 3180679334720)]


Compares equality across full and lean points, lists, tuples, and raw matrices using `==`, `.eq`, and `.eq_fast` with `point_spec` to benchmark validation/normalization pathways.

In [27]:
from upxo.geoEntities.point3d import Point3d as p3d

print(p3d(3, 4, 5) == p3d_leanest(3, 4, 5))
print(p3d(3, 4, 5) == [p3d_leanest(1, 4, 5), p3d_leanest(3, 4, 5)])
print(p3d(3, 4, 5) == p3d(3, 4, 5))
print(p3d(3, 4, 5) == [p3d(1, 2, 5), p3d(3, 4, 5)])
print(p3d(3, 4, 5) == (p3d(1, 4, 5), p3d(3, 4, 5)))
print(p3d(3, 4, 5) == [[1, 2, 5], [3, 4, 5], [5, 6, 5]])
print(p3d(3, 4, 5) == [[1, 3, 5], [2, 4, 6]])
print(p3d(3, 4, 5) == [[3, 4, 5]])
print(p3d(3, 4, 5) == [[3], [4], [5]])
print('##########################')
print(p3d(3, 4, 5).eq(p3d_leanest(3, 4, 5)))
print(p3d(3, 4, 5).eq([p3d_leanest(1, 4, 5), p3d_leanest(3, 4, 5)]))
print(p3d(3, 4, 5).eq(p3d(3, 4, 5)))
print(p3d(3, 4, 5).eq([p3d(1, 2, 5), p3d(3, 4, 5)]))
print(p3d(3, 4, 5).eq((p3d(1, 4, 5), p3d(3, 4, 5))))
print(p3d(3, 4, 5).eq([[1, 2, 5], [3, 4, 5], [5, 6, 5]]))
print(p3d(3, 4, 5).eq([[1, 3, 5], [2, 4, 6]]))
print(p3d(3, 4, 5).eq([[3, 4, 5]]))
print(p3d(3, 4, 5).eq([[3], [4], [5]]))
print('##########################')
print(p3d(3,4,5).eq_fast(p3d(3,4,5), point_spec=1))
print(p3d(3,4,5).eq_fast([p3d(1,2,5), p3d(3,4,5)], point_spec=2))
print(p3d(3,4,5).eq_fast((p3d(1,4,5), p3d(3,4,5)), point_spec=2))
print(p3d(3,4,5).eq_fast(p3d_leanest(3,4,5), point_spec=3))
print(p3d(3,4,5).eq_fast([p3d_leanest(1,4,5), p3d_leanest(3, 4, 5)], point_spec=4))
print(p3d(3,4,5).eq_fast([1,2,5], point_spec=5))
print(p3d(3,4,5).eq_fast([[1,2,5]], point_spec=6))
print(p3d(3,4,5).eq_fast([[1,2,5],[3,4,5],[5,6,5]], point_spec=7))
print(p3d(3,4,5).eq_fast([[3],[4],[5]], point_spec=8))
print(p3d(3,4,5).eq_fast([[3,3,4],[4,4,4],[5,5,5]], point_spec=8))

[True]
[False, True]
[True]
[False, True]
[False, True]
[False, True, False]
[False, False]
[True]
[True]
##########################
[True]
[False, True]
[True]
[False, True]
[False, True]
[False, True, False]
[False, False]
[True]
[True]
##########################
[True]
[False, True]
[False, True]
[True]
[False, True]
[False]
[False]
[False, True, False]
[True]
[True, True, False]


### from_three_planes

Find the intersection of three planes via `Point3d.from_three_planes`, highlighting how plane normals and anchor points affect solvability and what the returned point represents.

In [28]:
from upxo.geoEntities.plane import Plane

plane1 = Plane(point=(0, 0, 1), normal=(1, 1, 1))
plane2 = Plane(point=(0, 0, 0), normal=(0, 1, 0))
plane3 = Plane(point=(0, 0, 0), normal=(0, 0, 1))
intersection_point = p3d.from_three_planes(plane1, plane2, plane3)
print(intersection_point)

uxpo-p3d (1.0,0.0,0.0)


### squared_distance
#### Explanation
Calculate the squared distances between self point and plist, having
a list of 3D points.

#### Parameters
- plist: List of point objects
- point_spec: Integer representaing the type of point specification

#### DEVELOPMENT PHASES AND PROGRESS
- PHASE 1: 2D case: DONE
- PHASE 2: 3D case: DONE
- PHASE 3: Improve validations

Benchmark `Point3d.squared_distance` for varied input specs (single point, lean point, tuple, list-of-lists, large numpy arrays) to see how `point_spec` gates validation and vectorization.

In [29]:
print(p3d(0,0,0).squared_distance(p3d(1,1,0), point_spec=-1))
print(p3d(0,0,0).squared_distance([[1,2,3,4],[1,2,3,4],[1,2,3,4]], point_spec=-1))
points = np.random.random((3, 100000))
print(p3d(0,0,0).squared_distance(points))

print(p3d(0,0,0).squared_distance(p3d(1,1,0), point_spec=1))
print(p3d(0,0,0).squared_distance([p3d(1,1,0)], point_spec=2))
print(p3d(0,0,0).squared_distance(p3d_leanest(1,1,0), point_spec=3))
print(p3d(0,0,0).squared_distance([p3d_leanest(1,1,0)], point_spec=4))
print(p3d(0,0,0).squared_distance((1,1,0), point_spec=5))
print(p3d(0,0,0).squared_distance([(1,1,0)], point_spec=6))
print(p3d(0,0,0).squared_distance([[1,2,3],[4,5,6],[7,8,9]], point_spec=7))
print(p3d(0,0,0).squared_distance([[1,2,3,4],[1,2,3,4],[1,2,3,4]], point_spec=8))
print(p3d(0,0,0).squared_distance(np.random.random((3, 100000)), point_spec=8))

[2]
[ 3 12 27 48]
[0.22402853 1.54798099 0.56336461 ... 0.28243447 1.20200376 1.25529222]
2
2
2
2
2
2
[ 14  77 194]
[ 3 12 27 48]
[0.36510785 1.02695525 0.87167317 ... 0.77228522 1.49120154 0.52381163]


### distance
#### Exaplanation
Calculate the distances between self point and plist.

Compute true Euclidean `distance` with the same shapes as above to contrast with squared distances and note any precision/performance implications when toggling `point_spec`.

In [30]:
print(p3d(0,0,0).distance(p3d(1,1,0), point_spec=-1))
print(p3d(0,0,0).distance([[1,2,3,4],[1,2,3,4],[1,2,3,4]], point_spec=-1))
points = np.random.random((3, 100000))
print(p3d(0,0,0).distance(points))
print(p3d(0,0,0).distance(p3d(1,1,0), point_spec=1))
print(p3d(0,0,0).distance([p3d(1,1,0)], point_spec=2))
print(p3d(0,0,0).distance(p3d_leanest(1,1,0), point_spec=3))
print(p3d(0,0,0).distance([p3d_leanest(1,1,0)], point_spec=4))
print(p3d(0,0,0).distance((1,1,0), point_spec=5))
print(p3d(0,0,0).distance([(1,1,0)], point_spec=6))
print(p3d(0,0,0).distance([[1,2,3],[4,5,6],[7,8,9]], point_spec=7))
print(p3d(0,0,0).distance([[1,2,3,4],[1,2,3,4],[1,2,3,4]], point_spec=8))
print(p3d(0,0,0).distance(np.random.random((3, 100000)), point_spec=8))

[1.41421356]
[1.73205081 3.46410162 5.19615242 6.92820323]
[0.88384386 1.36604376 0.95093551 ... 1.02260866 0.87120832 0.45350973]
1.4142135623730951
1.4142135623730951
1.4142135623730951
1.4142135623730951
1.4142135623730951
1.4142135623730951
[ 3.74165739  8.77496439 13.92838828]
[1.73205081 3.46410162 5.19615242 6.92820323]
[1.2879982  0.37935437 1.14903113 ... 1.29854876 1.06593679 1.00493988]


### translate
Translate the self along the vector by dist.

Translates a `Point3d` along a direction vector by a specified distance, showing in-place mutation (`update=True`) and tolerance for non-throwing operations.

In [31]:
A = p3d(0, 0, 0)
A.translate(vector=[-1,-1,-1], dist=3.4641016151377544, update=True, throw=False)
print(A)

uxpo-p3d (-2.0,-2.0,-2.0)


### find_neigh_point_by_count

Calls `find_neigh_point_by_count` on mixed 2D coords projected onto `plane='xy'` to fetch k-nearest neighbors, illustrating cross-dimension handling between 3D host and 2D inputs.

In [32]:
from upxo.geoEntities.point2d import Point2d as p2d
p3d(0,0,0).find_neigh_point_by_count( plist=[[1, 2], [10, 12], [0, -5], [0, 0]], n=2, plane='xy')

(array([0, 3]),)